# Qwen3 Embedding 0.6B INT4 AWQ

This notebook clones `awq-embed`, installs the quantization-only runtime, runs AWQ search on local dummy calibration text, and saves packed INT4 weights for `Qwen/Qwen3-Embedding-0.6B`.

In [ ]:
import torch

print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("Enable a GPU runtime before running AWQ quantization.")
print("gpu", torch.cuda.get_device_name(0))
print("bf16 supported", torch.cuda.is_bf16_supported())

In [ ]:
!nvidia-smi

In [ ]:
%cd /content
![ -d awq-embed ] || git clone https://github.com/phunggiahuy159/awq-embed.git
%cd /content/awq-embed

!python -m pip install --upgrade pip
!python -m pip install -e . --no-deps
!python -m pip install "cachetools<7" "transformers>=4.51.0" "accelerate==0.34.2" datasets sentencepiece "tokenizers>=0.12.1" texttable toml attributedict protobuf tqdm scikit-learn

In [ ]:
%%bash
python - <<'PY'
import awq.entry
import awq.quantize.pre_quant
import awq.quantize.auto_scale
import awq.quantize.qmodule
print("AWQ imports succeeded without compiled kernels")
PY

In [ ]:
!python -m awq.entry \
  --model_path Qwen/Qwen3-Embedding-0.6B \
  --model_type embedding \
  --dtype bfloat16 \
  --w_bit 4 \
  --q_group_size 128 \
  --calib_data dummy \
  --calib_n_samples 16 \
  --calib_seqlen 128 \
  --run_awq \
  --dump_awq awq_cache/qwen3-embedding-0.6b-w4-g128-dummy.pt

In [ ]:
!python -m awq.entry \
  --model_path Qwen/Qwen3-Embedding-0.6B \
  --model_type embedding \
  --dtype bfloat16 \
  --w_bit 4 \
  --q_group_size 128 \
  --load_awq awq_cache/qwen3-embedding-0.6b-w4-g128-dummy.pt \
  --q_backend real \
  --dump_quant quant_cache/qwen3-embedding-0.6b-w4-g128-awq.pt

In [ ]:
from pathlib import Path

paths = [
    Path("awq_cache/qwen3-embedding-0.6b-w4-g128-dummy.pt"),
    Path("quant_cache/qwen3-embedding-0.6b-w4-g128-awq-v2.pt"),
]
for path in paths:
    if path.exists():
        print(f"{path}: {path.stat().st_size / (1024 ** 2):.2f} MiB")
    else:
        print(f"missing: {path}")

In [ ]:
# Fast MTEB-style classification probe: original vs AWQ pseudo-INT4.
# The packed INT4 .pt checkpoint is for kernel-backed inference; this cell uses
# fake/pseudo quantization so it can run in plain Colab PyTorch without building kernels.
import gc
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from datasets import load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from transformers import AutoModel, AutoTokenizer

from awq.quantize.pre_quant import apply_awq
from awq.quantize.quantizer import pseudo_quantize_model_weight

MODEL_ID = "Qwen/Qwen3-Embedding-0.6B"
AWQ_PATH = Path("awq_cache/qwen3-embedding-0.6b-w4-g128-dummy.pt")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, padding_side="left", trust_remote_code=True)
if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
    tokenizer.pad_token = tokenizer.eos_token

dataset = load_dataset("ag_news")
train_texts = dataset["train"]["text"][:512]
train_labels = dataset["train"]["label"][:512]
test_texts = dataset["test"]["text"][:256]
test_labels = dataset["test"]["label"][:256]

def last_token_pool(last_hidden_states, attention_mask):
    left_padding = attention_mask[:, -1].sum() == attention_mask.shape[0]
    if left_padding:
        return last_hidden_states[:, -1]
    sequence_lengths = attention_mask.sum(dim=1) - 1
    batch_size = last_hidden_states.shape[0]
    return last_hidden_states[torch.arange(batch_size, device=last_hidden_states.device), sequence_lengths]

@torch.no_grad()
def encode_texts(model, texts, batch_size=16, max_length=256):
    embeddings = []
    model.eval()
    for start in range(0, len(texts), batch_size):
        batch = texts[start : start + batch_size]
        encoded = tokenizer(batch, padding=True, truncation=True, max_length=max_length, return_tensors="pt").to(DEVICE)
        outputs = model(**encoded)
        pooled = last_token_pool(outputs.last_hidden_state, encoded["attention_mask"])
        pooled = F.normalize(pooled, p=2, dim=1)
        embeddings.append(pooled.float().cpu().numpy())
    return np.concatenate(embeddings, axis=0)

def eval_embeddings(name, model):
    x_train = encode_texts(model, train_texts)
    x_test = encode_texts(model, test_texts)
    clf = LogisticRegression(max_iter=1000, random_state=0)
    clf.fit(x_train, train_labels)
    pred = clf.predict(x_test)
    acc = accuracy_score(test_labels, pred)
    print(f"{name} AG News accuracy: {acc:.4f}")
    return acc

original_model = AutoModel.from_pretrained(MODEL_ID, torch_dtype=DTYPE, trust_remote_code=True).to(DEVICE)
original_acc = eval_embeddings("Original", original_model)
del original_model
gc.collect()
torch.cuda.empty_cache()

awq_model = AutoModel.from_pretrained(MODEL_ID, torch_dtype=DTYPE, trust_remote_code=True).to("cpu")
awq_results = torch.load(AWQ_PATH, map_location="cpu")
apply_awq(awq_model, awq_results)
pseudo_quantize_model_weight(
    awq_model,
    w_bit=4,
    q_config={"zero_point": True, "q_group_size": 128},
)
awq_model = awq_model.to(DEVICE)
awq_acc = eval_embeddings("AWQ pseudo-INT4", awq_model)
print(f"Accuracy delta: {awq_acc - original_acc:+.4f}")
